# Снижение активности по картам: кто, насколько и за счёт чего

**Автор:**
**Период данных:** 2024-01-01 — 2025-06-30
**Подготовлено для:** департамент розничных продуктов, банк «Меридиан»

<!--
Этот ноутбук собирается в HTML командой:
jupyter nbconvert --to html --no-input --execute 02_report.ipynb --output report.html

Поэтому: кода читатель не видит, все графики и таблицы получены
исполнением, числа в тексте должны совпадать с выводом кода.
Лимит: 6 страниц и 8 графиков.
-->

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

ROOT_PATH = Path().resolve().parent
DATA_PATH = ROOT_PATH / "data"
CLEAN = DATA_PATH / "clean_data.csv"
MOTNHLY = DATA_PATH / "monthly_client_activity.csv"
CAMPAIGN = ROOT_PATH / "campaign_list.csv"
LOG = ROOT_PATH / "cleaning_log.csv"

clean = pd.read_csv(CLEAN)
monthly = pd.read_csv(MOTNHLY)
campaign = pd.read_csv(CAMPAIGN)
log = pd.read_csv(LOG)

## 1. Главное

<!-- 3–5 выводов с числами. Пишется последним, читается первым.
     Каждый вывод — утверждение, а не тема. Не «анализ активности»,
     а «активность упала на X% у группы Y за счёт Z». -->

1. В справочнике 5625 уникальных клиентов, из них у 5245 есть хотя бы одна успешная операция в 2024 году — только для них можно сравнивать процентное изменение.

2. Средняя частота успешных операций снизилась с 6,00 до 4,92 операции на клиента в месяц (−17,9% при сравнении средних по всей клиентской базе). Медианное индивидуальное изменение среди клиентов с базовой активностью — снижение на 18,4%.

3. Наиболее выраженное падение наблюдается в массовом сегменте (медианное снижение 33,3%) и среди клиентов без мобильного приложения (27,5% против 14,6% у пользователей приложения).

4. Предлагается кампания из 1 337 клиентов в трёх непересекающихся группах: 691 + 154 + 492. Все имеют согласие на маркетинговую коммуникацию. Минимальная группа — 154 человека.

5. Главные сдвиги: частота операций по категории «Одежда и обувь» снизилась на 31,0%, по СБП — на 28,5%, по пластиковой карте — на 26,9%. При этом токенизированные операции выросли на 28,0%.

## 2. Контекст

<!-- Какое решение поддерживает этот отчёт, 3–5 предложений. -->

Банк наблюдает сокращение использования карт и может вернуть активность только 2 000 клиентам. Задача — выделить несколько групп с понятным профилем и причиной падения, чтобы под каждую группу подготовить своё решение. В итоговый список включены только клиенты, согласные на маркетинговую коммуникацию.

## 3. Данные и ограничения

<!-- Период, число клиентов и операций, что и почему исключено из расчёта,
     чего в данных нет.

     Здесь же одним абзацем: какую метрику активности вы выбрали,
     как она считается и какое снижение она не увидит. -->

## 3. Данные и ограничения

В исследовании использованы два исходных файла: справочник клиентов
и журнал операций за период с 01.01.2024 по 30.06.2025.

Справочник содержит 5 680 строк и 5 625 уникальных клиентов.
После удаления повторных записей для каждого клиента оставлена
одна строка. При наличии конфликтующих значений сохранена первая
исходная запись; 15 идентификаторов с конфликтами не исправлялись
предположениями.

В журнале операций первоначально 623 089 строк. После удаления
1 500 повторных записей по `txn_id` осталось 621 589 операций.
86 операций находятся вне целевого периода. После ограничения
периода осталось 621 503 операции.

Далее исключены 32 067 операций, относящихся к 375 уникальным
идентификаторам клиентов, отсутствующим в справочнике.
Из оставшихся 589 436 операций 18 434 имеют статус «отклонено»
или «возврат». В расчёте активности учитываются только успешные
операции — 571 002 записи.

**Метрика активности.** Используется среднее число успешных
операций клиента в месяц. Для базового периода количество
успешных операций делится на 12 месяцев 2024 года, для
оцениваемого периода — на 6 месяцев января–июня 2025 года.
Учитываются все валюты и каналы оплаты: карта, токен, СБП
и онлайн-банк. Суммы разных валют не складываются.

Таким образом, метрика отражает частоту успешных операций
в платёжных каналах банка, а не исключительно использование
пластиковой карты. Она не обнаружит снижение оборота или
среднего чека при неизменном количестве операций, а также
не отражает количество отклонённых попыток оплаты.

В данных отсутствуют сведения о причинах изменения поведения
клиентов, их фактических расходах в других банках и у других
платёжных провайдеров, а также информация о том, почему клиент
не отвечает на маркетинговые опросы.

## 4. Находки

<!-- По одной находке на подраздел. Заголовок подраздела — вывод,
     а не тема. Под заголовком: график или таблица, затем 2–4
     предложения. -->

In [ ]:
n_clients = len(clean)
n_base = int((clean.base_avg > 0).sum())
base_mean = clean.base_avg.mean()
eval_mean = clean.eval_avg.mean()
overall_change = 1 - eval_mean/base_mean
median_drop = clean.loc[clean.base_avg>0,"drop_pct"].median()

print(f"Клиентов: {n_clients:,}; с базовой активностью: {n_base:,}")
print(f"Среднее число операций/мес.: {base_mean:.2f} → {eval_mean:.2f} ({overall_change:.1%})")
print(f"Медианное индивидуальное падение: {median_drop:.1%}")

### 4.1. Снижение неравномерно распределено между сегментами

Массовый сегмент показывает наиболее сильное падение: медианное снижение — 33,3%, тогда как в зарплатном сегменте — 5,7%. Это поддерживает приоритет массового сегмента, но не доказывает, что именно тип сегмента вызывает снижение.

In [ ]:
seg = clean.groupby("segment").agg(
    count=("client_id","size"), base=("base_avg","mean"), eval=("eval_avg","mean"),
    median_drop=("drop_pct","median")
).sort_values("median_drop",ascending=False)
fig, ax = plt.subplots(figsize=(8,3.8))
ax.bar(seg.index,seg.median_drop*100)
ax.set(title="Медианное индивидуальное снижение по сегментам",
       xlabel="Сегмент",ylabel="Снижение, %")
ax.tick_params(axis="x",rotation=15)
for i,v in enumerate(seg.median_drop*100): ax.text(i,v+1,f"{v:.1f}%",ha="center",fontsize=9)
plt.tight_layout(); plt.show()
display(seg.round(3))

### 4.2. Отсутствие приложения связано с более сильным падением

У клиентов без приложения медианное снижение составляет 27,5%, у клиентов с приложением — 14,6%. Разница описательная: по этим данным нельзя утверждать, что установка приложения сама по себе восстановит активность.

In [ ]:
app = clean.groupby("has_mobile_app").agg(
    n=("client_id","size"), base=("base_avg","mean"), eval=("eval_avg","mean"),
    median_drop=("drop_pct","median")
)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.bar(app.index,app.median_drop*100)
ax.set(title="Медианное снижение по наличию приложения",
       xlabel="Мобильное приложение",ylabel="Снижение, %")
for i,v in enumerate(app.median_drop*100): ax.text(i,v+1,f"{v:.1f}%",ha="center")
plt.tight_layout(); plt.show()
display(app.round(3))

### 4.3. Падение сосредоточено в одежде/обуви и традиционных каналах оплаты

По категориям наибольшее снижение среднемесячного числа операций — в «Одежде и обуви» (31,0%). По каналам сильнее всего сократились СБП (28,5%) и пластиковая карта (26,9%). Одновременно токенизированные операции выросли на 28,0%: часть динамики может отражать смену способа оплаты, а не полный уход клиента из банка.

In [ ]:
def dimension_change(prefix):
    rows=[]
    for col in clean.columns:
        if col.startswith(prefix) and col.endswith("_base"):
            label=col[len(prefix):-len("_base")]
            b=clean[col].sum()
            e=clean[prefix+label+"_eval"].sum()
            rows.append([label,b,e,1-e/b if b else np.nan])
    return pd.DataFrame(rows,columns=["Признак","2024, операций/мес.","Янв–июн 2025, операций/мес.","Изменение"])
cat_change=dimension_change("cat_")
ch_change=dimension_change("ch_")
fig,ax=plt.subplots(figsize=(8,4))
vals=cat_change.sort_values("Изменение")
ax.barh(vals["Признак"],vals["Изменение"]*100)
ax.set(title="Изменение частоты по категориям",xlabel="Снижение, %",ylabel="Категория")
plt.tight_layout();plt.show()
display(cat_change.sort_values("Изменение",ascending=False).round(3))
fig,ax=plt.subplots(figsize=(7,3.5))
vals=ch_change.sort_values("Изменение")
ax.barh(vals["Признак"],vals["Изменение"]*100)
ax.set(title="Изменение частоты по каналам",xlabel="Снижение, % (отрицательное значение — рост)",ylabel="Канал")
plt.tight_layout();plt.show()
display(ch_change.sort_values("Изменение",ascending=False).round(3))

### 4.4. Динамика по месяцам подтверждает более низкий уровень в 2025 году

Сравнение по месяцам показывает сезонность внутри 2024 года (в частности, декабрьский максимум), поэтому отдельный месяц нельзя напрямую сравнивать с другим. Среднее по полным окнам — более надёжная опора для вывода.

In [ ]:
month_cols=[c for c in monthly.columns if c!="client_id"]
monthly_totals=monthly[month_cols].sum()
monthly_avg=monthly_totals/len(clean)
fig,ax=plt.subplots(figsize=(9,3.7))
ax.plot(month_cols,monthly_avg.values,marker="o")
ax.axvline(11.5,color="grey",linestyle="--")
ax.set(title="Успешные операции на клиента по месяцам",
       xlabel="Месяц",ylabel="Операций на клиента")
ax.tick_params(axis="x",rotation=45)
plt.tight_layout();plt.show()
display(monthly_avg.rename("Операций на клиента").round(2).to_frame())

## 5. Группы для кампании

<!-- Таблица: группа, правило отбора, размер, причина падения.
     Ниже — по абзацу на группу. -->

Правила ниже применяются к клиентам с согласием на маркетинговую коммуникацию. Для разрешения пересечений используется приоритет A → B → C; каждый клиент попадает максимум в одну группу. В выборке 1 337 клиентов из лимита 2 000.

In [ ]:
group_summary=campaign.merge(clean,on="client_id").groupby("group").agg(
    n=("client_id","nunique"),base=("base_avg","mean"),eval=("eval_avg","mean"),
    median_drop=("drop_pct","median")
)
group_summary["change"]=1-group_summary["eval"]/group_summary["base"]
display(group_summary.round(3))
assert campaign.client_id.is_unique
assert len(campaign)<=2000
assert group_summary["n"].min()>=150

A. Массовый сегмент без приложения — 691 клиент. Правило: segment = Массовый, has_mobile_app = Нет, базовая частота не ниже 1 успешной операции в месяц и общее снижение среднемесячной частоты не менее 20%. В отобранной группе средняя частота упала с 5,75 до 2,38 операции/мес. (−58,6%). Снижение затронуло все категории; по СБП падение составило около 62%. Рекомендуемый сценарий — простая коммуникация о доступных способах оплаты и цифровых сервисах, без предположения, что установка приложения автоматически вернёт все операции.

B. Клиенты с высокой долей покупок одежды и резким падением этой категории — 154 клиента. Правило: в 2024 году «Одежда и обувь» составляла не менее 25% успешных операций; среднемесячная частота этой категории упала не менее чем на 40%, общая частота — не менее чем на 20%, базовая частота — не менее 2 операций/мес.; клиенты группы A исключаются. В группе частота по одежде/обуви упала на 92% (с 344 до 28 операций в месяц на всю группу), общая — на 39,4%. Размер всего на 4 человека выше минимального порога 150, поэтому при новой выгрузке его нужно пересчитать. Возможный сценарий — предложение, связанное с категорией, а не универсальное сообщение о карте.

C. Клиенты, ранее преимущественно платившие пластиковой картой, — 492 клиента. Правило: не менее 60% базовых операций проходило по каналу «Карта»; частота по этому каналу снизилась не менее чем на 40%, общая частота — не менее чем на 20%, базовая частота — не менее 2 операций/мес.; клиенты A и B исключаются. Общая частота упала на 52,0%, а канал «Карта» — на 72%; при этом токенизированные операции выросли. Сценарий — помощь с переходом на удобный способ оплаты и проверка, не связана ли динамика с заменой пластика на токен.

## 6. Кого не включаем и почему

<!-- Отдельным списком. Для каждой категории: кто это, сколько их,
     почему кампания им не нужна или бесполезна. -->

- 646 клиентов без согласия на маркетинг — исключены независимо от степени снижения; контактировать с ними в рамках кампании нельзя.
- 380 клиентов без успешных операций в 2024 году — нет положительной базовой частоты, поэтому процентное падение не определено.
- Клиенты, не попавшие в три правила — не включены, поскольку их падение не связано с достаточно конкретным профилем/сценарием коммуникации в рамках лимита. Это не означает отсутствия у них проблем.
- Клиенты, уже показывающие рост активности, не являются приоритетом кампании возврата активности.

Категории исключений могут пересекаться: один клиент может
одновременно не иметь базовой активности и не иметь согласия
на коммуникацию. Поэтому приведённые количества не следует
складывать для получения общего числа исключённых клиентов.

## 7. Что проверили и что не подтвердилось

<!-- 3–5 строк, по одной на гипотезу, каждая с числом. -->

«Падают все сегменты одинаково» — не подтвердилось: медианное снижение от 5,7% в зарплатном до 33,3% в массовом сегменте.
«Отсутствие приложения не связано с падением» — не подтвердилось: 27,5% медианного снижения без приложения против 14,6% с приложением; причинность не установлена.
«Все каналы снижаются» — не подтвердилось: токенизированные операции выросли на 28,0%, тогда как карта и СБП снизились.
«Главный спад одинаков во всех категориях» — не подтвердилось: одежда/обувь снизилась на 31,0%, кафе/рестораны — на 12,4%.
«MCC надёжно подтверждает категорию» — не подтвердилось: в каждой из 9 нормализованных категорий встречаются все 9 MCC; MCC не использовался для категоризации.

## 8. Приложение

### 8.1. Сводка журнала изменений

<!-- 5–10 строк по типам нарушений: поле, что найдено, сколько строк,
     что сделано. Полный журнал — в cleaning_log.csv. -->

In [ ]:
display(log[["field","issue","n_rows","action"]])

### 8.2. Методика

<!-- Точные формулы, фильтры, пороги и их значения. Всё, что нужно,
     чтобы повторить расчёт. -->

- Клиентский справочник: одна строка на client_id; при конфликте оставлена первая запись исходного файла. Неоднозначные 15 идентификаторов не исправлялись предположениями.
- Операции: одна строка на txn_id; если повтор отличается только полнотой времени, оставлена запись с наиболее точной отметкой времени.
- Окна: базовое — 01.01.2024–31.12.2024 (12 месяцев); оцениваемое — 01.01.2025–30.06.2025 (6 месяцев).
- В метрике: только status = успешно, клиент должен присутствовать в справочнике, дата должна входить в заданный период. Счётчик операций не зависит от валюты; денежные суммы между валютами не суммируются.
- Для клиента: base_avg = число успешных операций в 2024 / 12; eval_avg = число успешных операций в январе–июне 2025 / 6; drop_pct = 1 − eval_avg / base_avg, если base_avg > 0.
- Группа A: массовый сегмент, нет приложения, base_avg ≥ 1, drop_pct ≥ 0,20.
- Группа B: clothes_share ≥ 0,25, clothes_drop ≥ 0,40, drop_pct ≥ 0,20, base_avg ≥ 2, не A.
- Группа C: card_share ≥ 0,60, card_drop ≥ 0,40, drop_pct ≥ 0,20, base_avg ≥ 2, не A и не B.
- Финальный фильтр кампании: marketing_opt_in = Да. Лимиты проверены программно.
- `monthly_income`: 49 значений-маркеров `-999` заменены на `NaN`.
  Исходные пропуски не заполнялись. `branch_code` приведён к строковому
  типу без изменения значений.
- Выбросы сумм обнаруживались по правилу 1,5 IQR отдельно для каждой
  комбинации валюты и статуса операции. Обнаруженные значения оставлены,
  поскольку статистический критерий сам по себе не подтверждает ошибку.
- Описательные статистики: для возраста используются среднее,
  стандартное отклонение, медиана и межквартильный размах;
  для дохода — медиана и межквартильный размах; для порядковой оценки
  удовлетворённости — медиана, мода и частоты значений.

### 8.3. Использование ИИ

<!-- 3–5 строк: для чего использовали и что проверили сами. -->

ИИ применялся для черновых формулировок и проверки кода